# Polynomial LCG Equidistribution

Polynomial Linear Congruential Generators over GF(2).
Defined by $x_n = x_{n-q} \oplus (x_{n-q} \gg s)$
with characteristic polynomial $z^k + z^q + 1$.

In [ ]:
import regpoly._regpoly_cpp as _cpp
from regpoly.generateur import Generateur
from regpoly.combinaison import Combinaison
from regpoly.tests.equidistribution_test import (
    EquidistributionTest, METHOD_MATRICIAL
)

INT_MAX = 2**31 - 1

## 1. Create PolyLCG Generators

Parameters: `k` (degree), `q` (feedback tap), `s` (shift).

In [ ]:
polylcg_params = [
    {"k": 32, "q": 7,  "s": 3},
    {"k": 64, "q": 4,  "s": 7},
    {"k": 96, "q": 13, "s": 5},
]

generators = []
for p in polylcg_params:
    L = min(32, p["k"])
    g = Generateur(_cpp.create_generator("polylcg", p, L))
    generators.append(g)
    print(f"PolyLCG k={p['k']}, q={p['q']}, s={p['s']} -> k={g.k}, L={g.L}")

## 2. Equidistribution

In [ ]:
for g, p in zip(generators, polylcg_params):
    comb = Combinaison(J=1, Lmax=g.L)
    comb.components[0].add_gen(g)
    comb.reset()

    test = EquidistributionTest(L=g.L, delta=[INT_MAX]*(g.L+1),
                                mse=INT_MAX, method=METHOD_MATRICIAL)
    result = test.run(comb)

    print(f"\n=== PolyLCG k={p['k']}, q={p['q']}, s={p['s']} ===")
    print(f"Dimension gaps sum = {result.se}")
    result.display_table(comb, 'l')